# 🔬 UNet Crack Detection — Full Pipeline

This notebook contains the complete UNet-based crack segmentation pipeline,
organised in the order you should run the cells:

| # | Cell | Purpose |
|---|------|---------|
| 1 | Imports & Configuration | All libraries + user-editable paths/hyperparameters |
| 2 | Dataset Loader (`CrackDataset`) | Reads images & labels from disk |
| 3 | Data Augmentation (`transforms`) | Preprocessing for training and evaluation |
| 4 | UNet Architecture | The neural network definition |
| 5 | Training Script (`train`) | Trains the model and saves weights |
| 6 | Validation Script (`validation`) | Evaluates a saved `.pth` file on the test set |
| 7 | Prediction Script (`predict`) | Runs inference on new images, saves output masks |
| 8 | Compute Mean & Std (optional) | One-time utility to recalculate dataset statistics |

> **Tip:** Run cells **top to bottom** on every new session.
> Only Cell 1 needs editing — change the paths and hyperparameters there.

In [1]:
print("hello")

hello


In [3]:
# =============================================================================
# CELL 1 — IMPORTS & CONFIGURATION
# =============================================================================
# All third-party imports live here so they are easy to find and install.
# Edit the USER CONFIGURATION section below before running anything else.
# =============================================================================

# ── Standard library ─────────────────────────────────────────────────────────
import os
import time
import datetime
import random
from typing import Dict

# ── Numerical / image ────────────────────────────────────────────────────────
import numpy as np
from PIL import Image

# ── PyTorch core ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

# ── TorchVision ──────────────────────────────────────────────────────────────
from torchvision import transforms as tv_transforms
from torchvision.transforms import functional as TF
from torchvision import transforms as T

# ── Project utilities (must be in the same directory as this notebook) ────────
from src import UNet
from train_utils import train_one_epoch, evaluate, create_lr_scheduler

# =============================================================================
# USER CONFIGURATION  ← edit these values before running
# =============================================================================

# ── Paths ─────────────────────────────────────────────────────────────────────
# Root folder that contains training/ and test/ subfolders
DATA_PATH = r"C:\Users\imuba\Documents\GitHub\NILM\GRADUATE_EARLY_FILES\MY_FILES\Civil_Engineering\Code\3000lcrack"

# Folder where weight (.pth) files will be saved during training
WEIGHTS_SAVE_DIR = r"C:\Users\imuba\Documents\GitHub\NILM\GRADUATE_EARLY_FILES\MY_FILES\Civil_Engineering\Code\UNet\save_weights"

# ── For validation (Cell 6) ───────────────────────────────────────────────────
VALIDATION_WEIGHTS = r"C:\path\to\your\best_model.pth"   # ← update after training
VALIDATION_DATA    = DATA_PATH                                # usually the same root

# ── For prediction / inference (Cell 7) ──────────────────────────────────────
PREDICT_WEIGHTS    = r"C:\path\to\your\best_model.pth"   # ← update after training
PREDICT_INPUT_DIR  = r"C:\path\to\crack\images"          # folder of images to analyse
PREDICT_OUTPUT_DIR = r"C:\path\to\output\masks"          # where results are saved

# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_CLASSES  = 1        # number of foreground classes (crack = 1, background not counted)
BATCH_SIZE   = 8        # reduce to 4 if you get a CUDA out-of-memory error
EPOCHS       = 200      # number of full passes through the training data
LEARNING_RATE = 0.001   # initial learning rate (good default; lower to 0.0001 if unstable)
MOMENTUM     = 0.9      # SGD momentum — leave as-is
WEIGHT_DECAY = 1e-4     # L2 regularisation — leave as-is
SAVE_BEST    = False    # True = only keep the single best epoch; False = keep all
USE_AMP      = False    # True = mixed precision (faster on modern GPUs)
PRINT_FREQ   = 5        # how often (in steps) to print training progress

# ── Dataset statistics (computed once by Cell 8) ─────────────────────────────
# These values were calculated from the crack dataset using compute_mean_std.
# Only re-run Cell 8 if you switch to a completely different dataset.
MEAN = (0.709, 0.381, 0.224)
STD  = (0.127, 0.079, 0.043)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Create weights save directory if it doesn't exist
os.makedirs(WEIGHTS_SAVE_DIR, exist_ok=True)
print(f"Weights will be saved to: {WEIGHTS_SAVE_DIR}")

Using device: cuda
Weights will be saved to: C:\Users\imuba\Documents\GitHub\NILM\GRADUATE_EARLY_FILES\MY_FILES\Civil_Engineering\Code\UNet\save_weights


---
## Cell 2 — Dataset Loader (`CrackDataset`)

Reads crack images (`.jpg`) and their binary label masks (`.png`) from disk.

**Expected folder structure:**
```
DATA_PATH/
├── training/
│   ├── image/   ← .jpg original images used for training
│   └── label/   ← .png binary masks (white = crack, black = background)
└── test/
    ├── image/   ← .jpg original images used for validation/evaluation
    └── label/   ← .png binary masks (required for scoring)
```

> Label inversion: `(255 - pixel) / 255` converts white-crack labels to
> a 0/1 float mask that the model expects.

In [4]:
# =============================================================================
# CELL 2 — DATASET LOADER
# =============================================================================

class CrackDataset(Dataset):
    """
    PyTorch Dataset for crack segmentation.

    Loads paired (image, label) samples from a structured directory.
    Supports both training (with augmentation) and test (evaluation only) modes.

    Args:
        root (str): Root directory containing 'training/' and 'test/' subfolders.
        train (bool): If True, loads from 'training/'; otherwise from 'test/'.
        transforms: Optional composed transform pipeline (see Cell 3).
    """

    def __init__(self, root: str, train: bool, transforms=None):
        super(CrackDataset, self).__init__()

        # Determine which subfolder to use
        self.flag = "training" if train else "test"
        data_root = os.path.join(root, self.flag)

        assert os.path.exists(data_root), (
            f"Dataset folder not found: '{data_root}'\n"
            f"Make sure DATA_PATH is correct and the folder contains 'training/' and 'test/'."
        )

        self.transforms = transforms

        # Collect all .jpg filenames in the image/ subfolder
        img_names = [
            f for f in os.listdir(os.path.join(data_root, "image"))
            if f.endswith(".jpg")
        ]

        # Build full paths for images and their corresponding label masks
        self.img_list = [os.path.join(data_root, "image", i) for i in img_names]
        self.label_list = [
            os.path.join(data_root, "label", os.path.splitext(i)[0] + ".png")
            for i in img_names
        ]

        # Verify every label file exists before training starts
        for label_path in self.label_list:
            if not os.path.exists(label_path):
                raise FileNotFoundError(
                    f"Label not found: '{label_path}'\n"
                    f"Each image must have a matching .png label with the same base name."
                )

        print(f"[CrackDataset] Loaded {len(self.img_list)} samples from '{data_root}'")

    def __getitem__(self, idx):
        # Load image as RGB (3 channels)
        img = Image.open(self.img_list[idx]).convert("RGB")

        # Load label as grayscale, then invert and normalise to [0, 1]
        # Original labels: white (255) = crack, black (0) = background
        # After inversion: 1.0 = crack, 0.0 = background (what the model expects)
        label = Image.open(self.label_list[idx]).convert("L")
        label = np.clip((255 - np.array(label)) / 255.0, 0, 1)

        # Convert back to PIL so the transform pipeline (Cell 3) can process it
        label = Image.fromarray(label)

        if self.transforms is not None:
            img, label = self.transforms(img, label)

        return img, label

    def __len__(self):
        return len(self.img_list)

    @staticmethod
    def collate_fn(batch):
        """
        Custom collate function to handle images of different sizes in the same batch.
        Pads all images/labels to the largest size in the batch.
        Images are padded with 0 (black); labels are padded with 255 (ignored during loss).
        """
        images, targets = list(zip(*batch))
        batched_imgs    = CrackDataset._pad_batch(images, fill_value=0)
        batched_targets = CrackDataset._pad_batch(targets, fill_value=255)
        return batched_imgs, batched_targets

    @staticmethod
    def _pad_batch(tensors, fill_value=0):
        """Pad a list of tensors to the maximum spatial dimensions in the batch."""
        max_size    = tuple(max(s) for s in zip(*[t.shape for t in tensors]))
        batch_shape = (len(tensors),) + max_size
        batched     = tensors[0].new(*batch_shape).fill_(fill_value)
        for t, padded in zip(tensors, batched):
            padded[..., :t.shape[-2], :t.shape[-1]].copy_(t)
        return batched

---
## Cell 3 — Data Augmentation & Preprocessing (`transforms`)

Defines all image transformations applied before feeding data to the model.

| Transform | Applied During | Purpose |
|-----------|---------------|---------|
| `RandomResize` | Training only | Scale image randomly between 50%–120% of base size — teaches scale invariance |
| `RandomHorizontalFlip` | Training only | 50% chance of left-right flip — prevents directional bias |
| `RandomVerticalFlip` | Training only | 50% chance of vertical flip — handles upside-down cracks |
| `RandomCrop(480)` | Training only | Extract random 480×480 patch — exposes model to different regions |
| `ToTensor` | Always | Convert PIL image to PyTorch tensor; scales pixels from [0,255] → [0,1] |
| `Normalize` | Always | Subtract channel mean, divide by std — centres data for stable training |

> Augmentation only applies during **training**. Validation and prediction use `ToTensor` + `Normalize` only.

In [5]:
# =============================================================================
# CELL 3 — DATA AUGMENTATION & PREPROCESSING
# =============================================================================

def pad_if_smaller(img: Image.Image, size: int, fill: int = 0) -> Image.Image:
    """
    If the image's shortest side is smaller than `size`, pad it with `fill`
    so that RandomCrop can always extract a valid crop.
    """
    min_side = min(img.size)
    if min_side < size:
        ow, oh = img.size
        pad_h = size - oh if oh < size else 0
        pad_w = size - ow if ow < size else 0
        img = TF.pad(img, (0, 0, pad_w, pad_h), fill=fill)
    return img


class Compose:
    """Chain multiple (image, target) transforms together."""
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target


class RandomResize:
    """
    Resize both image and target mask to a random size between min_size and max_size.
    Uses nearest-neighbour interpolation for the mask to avoid creating invalid class values.
    """
    def __init__(self, min_size: int, max_size: int = None):
        self.min_size = min_size
        self.max_size = max_size if max_size is not None else min_size

    def __call__(self, image, target):
        size = random.randint(self.min_size, self.max_size)
        image  = TF.resize(image, size)
        target = TF.resize(target, size, interpolation=T.InterpolationMode.NEAREST)
        return image, target


class RandomHorizontalFlip:
    """Flip image and mask horizontally with probability `flip_prob`."""
    def __init__(self, flip_prob: float):
        self.flip_prob = flip_prob

    def __call__(self, image, target):
        if random.random() < self.flip_prob:
            image  = TF.hflip(image)
            target = TF.hflip(target)
        return image, target


class RandomVerticalFlip:
    """Flip image and mask vertically with probability `flip_prob`."""
    def __init__(self, flip_prob: float):
        self.flip_prob = flip_prob

    def __call__(self, image, target):
        if random.random() < self.flip_prob:
            image  = TF.vflip(image)
            target = TF.vflip(target)
        return image, target


class RandomCrop:
    """
    Extract a random (size × size) crop from both image and mask.
    Pads first if either dimension is smaller than `size`.
    Mask padding uses 255 (treated as 'ignore' during loss computation).
    """
    def __init__(self, size: int):
        self.size = size

    def __call__(self, image, target):
        image  = pad_if_smaller(image,  self.size, fill=0)
        target = pad_if_smaller(target, self.size, fill=255)
        crop_params = T.RandomCrop.get_params(image, (self.size, self.size))
        image  = TF.crop(image,  *crop_params)
        target = TF.crop(target, *crop_params)
        return image, target


class ToTensor:
    """
    Convert PIL image → float32 tensor in [0, 1].
    Convert mask → int64 tensor of class indices.
    """
    def __call__(self, image, target):
        image  = TF.to_tensor(image)
        target = torch.as_tensor(np.array(target), dtype=torch.int64)
        return image, target


class Normalize:
    """Normalise image channels using pre-computed dataset mean and std."""
    def __init__(self, mean, std):
        self.mean = mean
        self.std  = std

    def __call__(self, image, target):
        image = TF.normalize(image, mean=self.mean, std=self.std)
        return image, target  # target is unchanged — only image is normalised


# ── Preset pipelines ──────────────────────────────────────────────────────────

class SegmentationPresetTrain:
    """
    Full augmentation pipeline used during training.
    Applies random resize, flips, crop, tensor conversion, and normalisation.
    """
    def __init__(self, base_size: int = 565, crop_size: int = 480,
                 hflip_prob: float = 0.5, vflip_prob: float = 0.5,
                 mean=MEAN, std=STD):
        # Scale boundaries: 50% to 120% of base_size
        min_size = int(0.5 * base_size)   # too small → risks losing crack detail
        max_size = int(1.2 * base_size)   # too large → excessive memory use

        trans = [RandomResize(min_size, max_size)]
        if hflip_prob > 0:
            trans.append(RandomHorizontalFlip(hflip_prob))
        if vflip_prob > 0:
            trans.append(RandomVerticalFlip(vflip_prob))
        trans.extend([
            RandomCrop(crop_size),
            ToTensor(),
            Normalize(mean=mean, std=std),
        ])
        self.transforms = Compose(trans)

    def __call__(self, img, target):
        return self.transforms(img, target)


class SegmentationPresetEval:
    """
    Minimal pipeline used during validation and prediction.
    No augmentation — only tensor conversion and normalisation.
    """
    def __init__(self, mean=MEAN, std=STD):
        self.transforms = Compose([
            ToTensor(),
            Normalize(mean=mean, std=std),
        ])

    def __call__(self, img, target):
        return self.transforms(img, target)


def get_transform(train: bool, mean=MEAN, std=STD):
    """
    Returns the appropriate transform pipeline.

    Args:
        train (bool): True → augmented training pipeline;
                      False → minimal evaluation pipeline.
    """
    if train:
        return SegmentationPresetTrain(base_size=565, crop_size=480, mean=mean, std=std)
    else:
        return SegmentationPresetEval(mean=mean, std=std)

print("Transform classes defined.")

Transform classes defined.


---
## Cell 4 — UNet Architecture

The UNet is a U-shaped encoder–decoder network designed for image segmentation.

```
Input image (3 channels)
        │
   ┌────▼────┐   Encoder (downsampling path)
   │ DoubleConv│ ← base_c = 32 feature maps
   └────┬────┘
        │ skip connection ──────────────────────────────────────────┐
   ┌────▼────┐                                                      │
   │  Down×4  │ ← MaxPool + DoubleConv, doubles channels each step  │
   └────┬────┘                                                      │
        │       Bottleneck (deepest features, most compressed)      │
   ┌────▼────┐                                                      │
   │   Up×4   │ ← Upsample + concat skip connection + DoubleConv   ◄┘
   └────┬────┘
        │
   ┌────▼────┐
   │  OutConv │ ← 1×1 conv → num_classes (2: background + crack)
   └────┬────┘
        │
  Output mask (same spatial size as input)
```

**Key setting:** `base_c=32` (half the original 64 default) — faster training, less memory, sufficient for crack detection.

In [6]:
# =============================================================================
# CELL 4 — UNET ARCHITECTURE
# =============================================================================

class DoubleConv(nn.Sequential):
    """
    Two consecutive Conv2d → BatchNorm → ReLU blocks.
    The backbone building block used throughout the UNet.

    Args:
        in_channels:  Number of input feature channels.
        out_channels: Number of output feature channels.
        mid_channels: Intermediate channels (defaults to out_channels).
    """
    def __init__(self, in_channels: int, out_channels: int, mid_channels: int = None):
        if mid_channels is None:
            mid_channels = out_channels
        super().__init__(
            nn.Conv2d(in_channels,  mid_channels,  kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class Down(nn.Sequential):
    """
    Encoder step: MaxPool2d (halves spatial size) → DoubleConv (doubles channels).
    Used 4 times in the encoder path.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(
            nn.MaxPool2d(kernel_size=2, stride=2),
            DoubleConv(in_channels, out_channels),
        )


class Up(nn.Module):
    """
    Decoder step: upsample → concatenate skip connection → DoubleConv.

    Args:
        in_channels:  Input channels (from previous layer).
        out_channels: Output channels after DoubleConv.
        bilinear:     If True, use bilinear upsampling (smoother, fewer params).
                      If False, use transposed convolution (learns upsampling).
    """
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up   = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x1: Feature map from the previous decoder layer (smaller).
            x2: Skip connection feature map from the encoder (larger).
        """
        x1 = self.up(x1)

        # Pad x1 to match x2's spatial dimensions if there is a size mismatch
        # (can happen with non-power-of-2 input sizes)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,
                         diff_y // 2, diff_y - diff_y // 2])

        # Concatenate along channel dimension, then apply DoubleConv
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Sequential):
    """Final 1×1 convolution to map feature channels → class scores."""
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__(
            nn.Conv2d(in_channels, num_classes, kernel_size=1)
        )


class UNet(nn.Module):
    """
    Full UNet for semantic segmentation.

    Args:
        in_channels: Number of input image channels (3 for RGB).
        num_classes: Total output classes including background
                     (e.g. 2 = background + crack).
        bilinear:    Upsampling method — True = bilinear, False = transposed conv.
        base_c:      Base feature channel count. Doubles at each encoder step.
                     32 (used here) is lighter than the original paper's 64.
    """
    def __init__(self,
                 in_channels: int = 3,
                 num_classes: int = 2,
                 bilinear: bool   = True,
                 base_c: int      = 32):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.bilinear    = bilinear

        factor = 2 if bilinear else 1

        # Encoder (downsampling path) — channels: base_c → base_c*2 → *4 → *8 → *16
        self.in_conv = DoubleConv(in_channels, base_c)
        self.down1   = Down(base_c,      base_c * 2)
        self.down2   = Down(base_c * 2,  base_c * 4)
        self.down3   = Down(base_c * 4,  base_c * 8)
        self.down4   = Down(base_c * 8,  base_c * 16 // factor)   # bottleneck

        # Decoder (upsampling path) — mirrors encoder in reverse
        self.up1     = Up(base_c * 16, base_c * 8  // factor, bilinear)
        self.up2     = Up(base_c * 8,  base_c * 4  // factor, bilinear)
        self.up3     = Up(base_c * 4,  base_c * 2  // factor, bilinear)
        self.up4     = Up(base_c * 2,  base_c,                bilinear)

        # Output: produce one score map per class
        self.out_conv = OutConv(base_c, num_classes)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Returns:
            dict with key 'out' containing the raw logits tensor
            of shape (N, num_classes, H, W).
        """
        # Encoder
        x1 = self.in_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder — each Up layer receives (previous_decoder, skip_from_encoder)
        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)

        logits = self.out_conv(x)
        return {"out": logits}


def create_model(num_classes: int) -> UNet:
    """Instantiate a UNet with the project's standard settings."""
    return UNet(in_channels=3, num_classes=num_classes, base_c=32)

print("UNet architecture defined.")

UNet architecture defined.


---
## Cell 5 — Training

Trains the UNet model and saves weight files after each epoch.

- **Output:** `{WEIGHTS_SAVE_DIR}/model_{epoch}.pth` (or `best_model.pth` if `SAVE_BEST=True`)
- **Log:** `results_YYYYMMDD-HHMMSS.txt` with per-epoch loss, learning rate, and Dice score
- **Resume:** set the `--resume` argument to a `.pth` path to continue from a checkpoint

In [ ]:
# =============================================================================
# CELL 5 — TRAINING
# =============================================================================

def run_training():
    """
    Full training loop for the UNet crack detection model.

    Reads hyperparameters from Cell 1 (BATCH_SIZE, EPOCHS, LEARNING_RATE, etc.).
    Saves weight files and a metrics log to WEIGHTS_SAVE_DIR.
    """
    # Total classes = foreground classes + 1 background class
    num_classes = NUM_CLASSES + 1

    # Timestamped log file — one file per training run
    results_file = os.path.join(
        WEIGHTS_SAVE_DIR,
        "results_{}.txt".format(datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    )

    # ── Datasets ─────────────────────────────────────────────────────────────
    train_dataset = CrackDataset(DATA_PATH, train=True,
                                 transforms=get_transform(train=True))
    val_dataset   = CrackDataset(DATA_PATH, train=False,
                                 transforms=get_transform(train=False))

    # ── Data loaders ─────────────────────────────────────────────────────────
    # Use up to 8 worker processes for parallel data loading
    num_workers = min(os.cpu_count(), BATCH_SIZE if BATCH_SIZE > 1 else 0, 8)

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=num_workers,
        shuffle=True,          # randomise order each epoch
        pin_memory=True,       # speed up CPU→GPU transfers
        collate_fn=CrackDataset.collate_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=1,          # validate one image at a time
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=CrackDataset.collate_fn,
    )

    # ── Model ────────────────────────────────────────────────────────────────
    model = create_model(num_classes=num_classes)
    model.to(DEVICE)

    # Only pass parameters that actually require gradients to the optimiser
    params_to_optimise = [p for p in model.parameters() if p.requires_grad]

    # ── Optimiser ────────────────────────────────────────────────────────────
    # SGD with momentum — standard choice for segmentation tasks
    optimizer = torch.optim.SGD(
        params_to_optimise,
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    # Mixed-precision scaler (only active if USE_AMP=True)
    scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

    # ── Learning rate scheduler ───────────────────────────────────────────────
    # Updates LR every step (not every epoch); includes a linear warmup phase.
    # This prevents the model from making large, destabilising updates early on.
    lr_scheduler = create_lr_scheduler(
        optimizer, len(train_loader), EPOCHS, warmup=True
    )

    # ── Training loop ─────────────────────────────────────────────────────────
    best_dice  = 0.0
    start_time = time.time()

    for epoch in range(EPOCHS):
        # One full pass through the training set
        mean_loss, current_lr = train_one_epoch(
            model, optimizer, train_loader, DEVICE, epoch, num_classes,
            lr_scheduler=lr_scheduler, print_freq=PRINT_FREQ, scaler=scaler,
        )

        # Evaluate on validation set after each epoch
        confmat, dice = evaluate(model, val_loader, device=DEVICE, num_classes=num_classes)
        val_info = str(confmat)

        print(val_info)
        print(f"[Epoch {epoch:03d}] loss={mean_loss:.4f}  lr={current_lr:.6f}  dice={dice:.3f}")

        # Append results to the log file
        with open(results_file, "a") as f:
            f.write(
                f"[epoch: {epoch}]\n"
                f"train_loss: {mean_loss:.4f}\n"
                f"lr: {current_lr:.6f}\n"
                f"dice coefficient: {dice:.3f}\n"
                + val_info + "\n\n"
            )

        # Decide whether to save this epoch's weights
        if SAVE_BEST:
            if dice > best_dice:
                best_dice = dice
            else:
                continue  # skip saving if this epoch didn't improve

        # Build the checkpoint dictionary
        save_file = {
            "model":        model.state_dict(),
            "optimizer":    optimizer.state_dict(),
            "lr_scheduler": lr_scheduler.state_dict(),
            "epoch":        epoch,
        }
        if USE_AMP:
            save_file["scaler"] = scaler.state_dict()

        # Save to disk
        if SAVE_BEST:
            save_path = os.path.join(WEIGHTS_SAVE_DIR, "best_model.pth")
        else:
            save_path = os.path.join(WEIGHTS_SAVE_DIR, f"model_{epoch}.pth")

        torch.save(save_file, save_path)

    # ── Summary ───────────────────────────────────────────────────────────────
    elapsed = str(datetime.timedelta(seconds=int(time.time() - start_time)))
    print(f"\nTraining complete. Total time: {elapsed}")
    print(f"Results log: {results_file}")
    print(f"Weights saved to: {WEIGHTS_SAVE_DIR}")


# ── Run ───────────────────────────────────────────────────────────────────────
run_training()

---
## Cell 6 — Validation

Evaluates a saved `.pth` weight file against the labelled test set and prints accuracy metrics.

**Before running:** update `VALIDATION_WEIGHTS` in Cell 1 to point to your chosen `.pth` file.

**Metrics printed:**
- **Dice coefficient** — primary metric; higher is better (max 1.0)
- **IoU per class** — intersection over union for background and crack
- **Global pixel accuracy** — % of pixels correctly classified

In [ ]:
# =============================================================================
# CELL 6 — VALIDATION / EVALUATION
# =============================================================================

def run_validation():
    """
    Load a saved model checkpoint and evaluate it on the test set.

    Reads VALIDATION_WEIGHTS and VALIDATION_DATA from Cell 1.
    Prints Dice coefficient, IoU per class, and global pixel accuracy.
    """
    assert os.path.exists(VALIDATION_WEIGHTS), (
        f"Weight file not found: '{VALIDATION_WEIGHTS}'\n"
        "Update VALIDATION_WEIGHTS in Cell 1."
    )

    num_classes = NUM_CLASSES + 1

    # ── Dataset & loader ─────────────────────────────────────────────────────
    # Validation uses a fixed resize (not random) for reproducible evaluation
    class SegmentationEvalPreset:
        """Fixed-size evaluation preset: resize shortest side to base_size, then normalise."""
        def __init__(self, base_size: int = 565, mean=MEAN, std=STD):
            self.transforms = Compose([
                RandomResize(base_size, base_size),  # fixed size (min=max)
                ToTensor(),
                Normalize(mean=mean, std=std),
            ])
        def __call__(self, img, target):
            return self.transforms(img, target)

    val_dataset = CrackDataset(
        VALIDATION_DATA,
        train=False,
        transforms=SegmentationEvalPreset(),
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=1,
        num_workers=0,
        pin_memory=True,
        shuffle=False,
        collate_fn=CrackDataset.collate_fn,
    )

    # ── Load model ────────────────────────────────────────────────────────────
    model = UNet(in_channels=3, num_classes=num_classes, base_c=32)
    checkpoint = torch.load(VALIDATION_WEIGHTS, map_location=DEVICE)
    model.load_state_dict(checkpoint["model"])
    model.to(DEVICE)
    model.eval()

    print(f"Loaded weights from: {VALIDATION_WEIGHTS}")
    print(f"Evaluating on {len(val_dataset)} images...\n")

    # ── Evaluate ─────────────────────────────────────────────────────────────
    confmat, dice = evaluate(model, val_loader, device=DEVICE, num_classes=num_classes)

    print(confmat)
    print(f"Dice coefficient: {dice:.4f}")


# ── Run ───────────────────────────────────────────────────────────────────────
run_validation()

---
## Cell 7 — Prediction (Inference)

Runs the model on a folder of **unlabelled** crack images and saves output mask images.

**No labels needed** — this is the "real-world use" cell.

**Output format:** black pixels = crack, white pixels = background (same size as input).

**Before running:** update `PREDICT_WEIGHTS`, `PREDICT_INPUT_DIR`, and `PREDICT_OUTPUT_DIR` in Cell 1.

In [ ]:
# =============================================================================
# CELL 7 — PREDICTION / INFERENCE
# =============================================================================

def time_synchronized():
    """GPU-safe timing — waits for all CUDA kernels to finish before recording time."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.time()


def run_prediction():
    """
    Run crack detection inference on all images in PREDICT_INPUT_DIR.

    For each image:
      1. Preprocess (ToTensor + Normalize)
      2. Run UNet forward pass
      3. Convert logits → binary mask (crack=black, background=white)
      4. Save result as .png with the same base filename

    Reads PREDICT_WEIGHTS, PREDICT_INPUT_DIR, PREDICT_OUTPUT_DIR from Cell 1.
    """
    assert os.path.exists(PREDICT_WEIGHTS), (
        f"Weight file not found: '{PREDICT_WEIGHTS}'\n"
        "Update PREDICT_WEIGHTS in Cell 1."
    )
    assert os.path.exists(PREDICT_INPUT_DIR), (
        f"Input folder not found: '{PREDICT_INPUT_DIR}'\n"
        "Update PREDICT_INPUT_DIR in Cell 1."
    )

    os.makedirs(PREDICT_OUTPUT_DIR, exist_ok=True)

    # ── Preprocessing ────────────────────────────────────────────────────────
    # No augmentation — only normalise to match training distribution
    data_transform = tv_transforms.Compose([
        tv_transforms.ToTensor(),
        tv_transforms.Normalize(mean=MEAN, std=STD),
    ])

    # ── Load model ────────────────────────────────────────────────────────────
    model = UNet(in_channels=3, num_classes=NUM_CLASSES + 1, base_c=32)
    model.load_state_dict(torch.load(PREDICT_WEIGHTS, map_location="cpu")["model"])
    model.to(DEVICE)
    model.eval()  # disable dropout / batchnorm training behaviour

    print(f"Model loaded from: {PREDICT_WEIGHTS}")
    print(f"Reading images from: {PREDICT_INPUT_DIR}")
    print(f"Saving results to:   {PREDICT_OUTPUT_DIR}\n")

    # ── Collect input images ──────────────────────────────────────────────────
    supported_ext = (".png", ".jpg", ".jpeg", ".bmp", ".tiff")
    image_files = [
        f for f in os.listdir(PREDICT_INPUT_DIR)
        if f.lower().endswith(supported_ext)
    ]

    if not image_files:
        print("No images found in the input folder. Check PREDICT_INPUT_DIR.")
        return

    total_time = 0.0

    # ── Inference loop ────────────────────────────────────────────────────────
    for image_file in image_files:
        img_path = os.path.join(PREDICT_INPUT_DIR, image_file)

        # Load and preprocess
        original_img = Image.open(img_path).convert("RGB")
        img_tensor   = data_transform(original_img)
        img_tensor   = torch.unsqueeze(img_tensor, dim=0)  # add batch dimension

        with torch.no_grad():
            # Warm-up pass: initialises CUDA kernels so timing is accurate
            h, w = img_tensor.shape[-2:]
            model(torch.zeros((1, 3, h, w), device=DEVICE))

            # Timed inference pass
            t0     = time_synchronized()
            output = model(img_tensor.to(DEVICE))
            t1     = time_synchronized()

        elapsed     = t1 - t0
        total_time += elapsed

        # Convert raw logits → class index per pixel
        # argmax over channel dim: 0 = background, 1 = crack
        prediction = output["out"].argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

        # Map to output image:
        #   model class 0 (background) → white (255)
        #   model class 1 (crack)      → black (0)
        output_mask = np.zeros_like(prediction)
        output_mask[prediction == 0] = 255   # background → white
        output_mask[prediction == 1] = 0     # crack → black

        # Save result
        out_filename = os.path.splitext(image_file)[0] + ".png"
        out_path     = os.path.join(PREDICT_OUTPUT_DIR, out_filename)
        Image.fromarray(output_mask).save(out_path)

        print(f"  {image_file:40s}  inference: {elapsed:.3f}s  →  {out_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    n = len(image_files)
    print(f"\nDone. Processed {n} image(s).")
    print(f"Average inference time: {total_time / n:.3f}s per image.")


# ── Run ───────────────────────────────────────────────────────────────────────
run_prediction()

---
## Cell 8 — Compute Dataset Mean & Std *(optional, run once)*

Calculates the per-channel pixel mean and standard deviation across all training images.
The results are used by the `Normalize` transform in Cell 3.

**When to run this:**
- Only if you switch to a completely different dataset with different image characteristics.
- The values already in Cell 1 (`MEAN`, `STD`) were computed from the current crack dataset — do not change them unless you change the dataset.

**After running:** copy the printed values into `MEAN` and `STD` in Cell 1.

In [ ]:
# =============================================================================
# CELL 8 — COMPUTE DATASET MEAN & STD  (run once only)
# =============================================================================

def compute_mean_std(image_dir: str) -> None:
    """
    Compute per-channel mean and standard deviation for all images in a folder.

    Iterates over every .jpg in `image_dir`, accumulates statistics,
    and prints the mean and std to copy into Cell 1.

    Args:
        image_dir: Path to the folder containing .jpg training images.
    """
    assert os.path.exists(image_dir), f"Image directory not found: '{image_dir}'"

    img_paths = [
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith(".jpg")
    ]

    if not img_paths:
        print("No .jpg images found in the specified folder.")
        return

    num_channels    = 3
    cumulative_mean = np.zeros(num_channels)
    cumulative_std  = np.zeros(num_channels)

    for path in img_paths:
        # Normalise pixel values to [0, 1] before computing statistics
        img = np.array(Image.open(path).convert("RGB")) / 255.0
        cumulative_mean += img.mean(axis=(0, 1))   # mean over H and W
        cumulative_std  += img.std(axis=(0, 1))    # std over H and W

    mean = cumulative_mean / len(img_paths)
    std  = cumulative_std  / len(img_paths)

    print(f"Computed from {len(img_paths)} images:")
    print(f"  MEAN = ({mean[0]:.3f}, {mean[1]:.3f}, {mean[2]:.3f})")
    print(f"  STD  = ({std[0]:.3f},  {std[1]:.3f},  {std[2]:.3f})")
    print("\nCopy these values into MEAN and STD in Cell 1.")


# ── Run ───────────────────────────────────────────────────────────────────────
# Update this path to your training images folder
training_image_dir = os.path.join(DATA_PATH, "training", "image")
compute_mean_std(training_image_dir)